In [1]:
import sys
import os
sys.path.append(os.path.realpath('../../'))

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"
DEVICE = 'cuda:0'
import torch

In [3]:
from tqdm.auto import tqdm
from typing import List
import re
from typing import List, Tuple, Dict
from data.dataset import GraphDataset, ReimburseGraphDataset, DataAugmentationLevel, DialogNode, NodeType
import nltk
from statistics import mean 

In [4]:
# human_data_train = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/train_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../resources/")
# human_data_test = ReimburseGraphDataset('en/reimburse/test_graph.json', 'en/reimburse/test_answers.json', False, augmentation=DataAugmentationLevel.NONE, resource_dir="../../resources/")
generated_data = ReimburseGraphDataset('en/reimburse/train_graph.json', 'en/reimburse/generated/chatgpt/thesis/train_answers.json', False, augmentation=DataAugmentationLevel.ARTIFICIAL_ONLY, augmentation_path="en/reimburse/generated/chatgpt/thesis/train_questions_v3.json", resource_dir="../../resources/")

LOADING GRAPH...
- Loading questions from  ../../resources/en/reimburse/generated/chatgpt/thesis/train_questions_v3.json
GRAPH LOADED
LOADING ANSWERS FROM ../../resources/en/reimburse/generated/chatgpt/thesis/train_answers.json...
- not using synonyms
ANSWERS LOADED
===== Dataset Statistics =====
- files:  en/reimburse/train_graph.json en/reimburse/generated/chatgpt/thesis/train_answers.json
- synonyms: False
- depth: 20  - degree: 13
- answers: 81
- questions: 10281
- loaded original data: False
- loaded generated data: True
- question limit: 0  - maximum loaded:  200
- answer limit: 0  - maximum loaded:  1


In [5]:
from data.dataset import Question
from sentence_transformers import SentenceTransformer, util

similarity_model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2', device=DEVICE, cache_folder = '/mount/arbeitsdaten/asr-2/vaethdk/resources/weights').to(DEVICE)

In [6]:
def calculate_similarity(questions_1: List[Question], questions_2: List[Question]):
    #Compute embedding for both lists
    embeddings1 = similarity_model.encode([q.text.lower().strip() for q in questions_1], convert_to_tensor=True)
    embeddings2 = similarity_model.encode([q.text.lower().strip() for q in questions_2], convert_to_tensor=True)

    #Compute pair-wise cosine-similarities -> #questions_1 x #questions_2
    cosine_scores = util.cos_sim(embeddings1, embeddings2)
    return cosine_scores.to("cpu")


In [7]:
calculate_similarity([Question(key="a", text="He is king", parent=None)],[Question(key="a", text="He is a King", parent=None)])

tensor([[0.9469]])

In [8]:
def within_dataset_similarity(dataset: GraphDataset) -> List[float]:
    similarities = {}
    for node in tqdm(dataset.nodes_by_type[NodeType.INFO]):
        if len(node.questions) > 0:
            pariwise_sim = calculate_similarity(node.questions, node.questions)
            # ignore self-similarity (matrix diagnoal)
            # also: matrix is symmetric
            # --> only look at triangle above diagonal
            pariwise_sim = torch.triu(pariwise_sim, diagonal=1) # will set elements on and below diagonal to 0
            # pariwise_sim = pariwise_sim.view(-1) # convert to list
            # relevant_elements = pariwise_sim[pariwise_sim.nonzero()].view(-1) # extract nonzero entries
            similarities[node.key] = pariwise_sim 
    return similarities           

In [9]:
def question_node_similarity(dataset: GraphDataset) -> List[float]:
    similarities = {}
    for node in tqdm(dataset.nodes_by_type[NodeType.INFO]):
        if len(node.questions) > 0:
             #Compute embedding for both lists
            embeddings1 = similarity_model.encode(node.text, convert_to_tensor=True)
            embeddings2 = similarity_model.encode([q.text for q in node.questions], convert_to_tensor=True)

            cosine_scores = util.cos_sim(embeddings1, embeddings2).view(-1)
            similarities[node.key] = cosine_scores.tolist() 
    return similarities           

In [10]:
node_q_sim = question_node_similarity(generated_data)

  0%|          | 0/80 [00:00<?, ?it/s]

In [11]:
avg_sim_scores = []
for i in tqdm(range(3,201)):
    pairwise_sim_scores = []
    for node in generated_data.nodes_by_type[NodeType.INFO]:
        pairwise_sim_scores.extend(node_q_sim[node.key][:i])
    avg_sim_scores.append(mean(pairwise_sim_scores)) 

  0%|          | 0/198 [00:00<?, ?it/s]

In [25]:
import plotly.graph_objects as go

# ...existing code...
x_values = [3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 40, 50, 75, 100, 125, 150, 175, 200]
y_values = [avg_sim_scores[x-3] for x in x_values]
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=y_values, mode='lines+markers'))
fig.update_layout(
    xaxis_title="Number of Generated Questions",
    yaxis_title="Avg. Cosine Similarity",
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(size=26),
   
)
fig.update_xaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey', range=[0, 200], dtick=10)
fig.update_yaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey', dtick=0.02)
fig.show()

fig.write_image('question_node_similarities_200.pdf', 'pdf', width=1300, height=600)


In [15]:
generated_similarity = within_dataset_similarity(generated_data)

  0%|          | 0/80 [00:00<?, ?it/s]

In [152]:
avg_sim_scores = []
for i in tqdm(range(3,201)):
    pairwise_sim_scores = []
    for node in generated_data.nodes_by_type[NodeType.INFO]:
        submatrix = generated_similarity[node.key][:i, :i]
        relevant_elements = submatrix[submatrix != 0].reshape(-1)
        pairwise_sim_scores.extend(relevant_elements.tolist())
    avg_sim_scores.append(mean(pairwise_sim_scores)) 

  0%|          | 0/198 [00:00<?, ?it/s]

In [153]:
import plotly.graph_objects as go

# ...existing code...
x_values = list(range(3,201))
fig = go.Figure()
fig.add_trace(go.Scatter(x=x_values, y=avg_sim_scores, mode='lines+markers'))
fig.update_layout(
    title="Average Similarity Scores",
    xaxis_title="Index",
    yaxis_title="Score"
)
fig.show()

In [67]:
# count the number of unique quesitons per node
uniques = {}
num_nodes = len(generated_data.nodes_by_type[NodeType.INFO])
for i in tqdm(range(1,201)):
    avg_uniques_per_node = 0 
    for node in generated_data.nodes_by_type[NodeType.INFO]:
        node_uniques = len(set([q.text.lower().strip() for q in node.questions[:i]]))
        avg_uniques_per_node += node_uniques
    uniques[i] = avg_uniques_per_node / (num_nodes * i)
    

  0%|          | 0/200 [00:00<?, ?it/s]

In [85]:
import plotly.graph_objects as go

smaller_list_keys = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 30, 40, 50, 75, 100, 125, 150, 175, 200]
y = [uniques[key] * 100 for key in smaller_list_keys]

# ...existing code...
fig = go.Figure()
fig.add_trace(go.Scatter(x=smaller_list_keys, y=y, mode='lines+markers'))
fig.update_layout(
    title="Percent of unique questions",
    xaxis_title="#Questions",
    yaxis_title="Unique Questions [%]",
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(size=26),
   
)
fig.update_xaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey', range=[0, 200], dtick=10)
fig.update_yaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey')
fig.show()

fig.write_image('unique_questions_200.pdf', 'pdf', width=1300, height=600)

In [22]:
# import json
# with open("question_similarities_200.json", "r") as f:
#     data = json.load(f)

In [55]:
import math

In [146]:
threshold = 0.8
too_similar = {}
for i in tqdm(range(3,201)):
    avg_too_similar = []
    for node in generated_data.nodes_by_type[NodeType.INFO]:
        submatrix = generated_similarity[node.key][:i,:i]
        num_similar = (submatrix >= threshold).int().sum().item()
        avg_too_similar.append(num_similar / submatrix[submatrix != 0].size(0))
        assert  num_similar / submatrix[submatrix != 0].size(0) <= 1, f"{i}, {num_similar}, {(submatrix <= threshold)}, {submatrix}"
    too_similar[i] = mean(avg_too_similar)

  0%|          | 0/198 [00:00<?, ?it/s]

In [12]:
# find the node with the most duplicates
most_duplicate_node = None
most_duplicates = 0

for node in generated_data.nodes_by_type[NodeType.INFO]:
    node_uniques = len(set([q.text.lower().strip() for q in node.questions]))
    duplicates = len(node.questions) - node_uniques
    if duplicates > most_duplicates:
        most_duplicates = duplicates
        most_duplicate_node = node
print(most_duplicates)
print(most_duplicate_node.key)
    

158
16457022377469579


In [53]:
# find the node with the most duplicates
least_duplicate_node = None
least_duplicates = 201

for node in generated_data.nodes_by_type[NodeType.INFO]:
    node_uniques = len(set([q.text.lower().strip() for q in node.questions]))
    if least_duplicates > node_uniques:
        least_duplicates = node_uniques
        least_duplicate_node = node
print(least_duplicates)
print(least_duplicate_node.key)
    

5
16369793221368112


In [55]:
print(generated_similarity[most_duplicate_node.key][generated_similarity[most_duplicate_node.key] != 0].view(-1).mean())
print(generated_similarity[least_duplicate_node.key][generated_similarity[least_duplicate_node.key] != 0].view(-1).mean())

tensor(0.8500)
tensor(0.6855)


In [41]:
generated_similarity = within_dataset_similarity(generated_data)

  0%|          | 0/80 [00:00<?, ?it/s]

In [63]:
from collections import defaultdict

avg_sim_scores = {}
for i in tqdm(range(2,201)):
    pairwise_sim_scores = []
    for node in generated_data.nodes_by_type[NodeType.INFO]:
        submatrix = generated_similarity[node.key][:i, :i]
        relevant_elements = submatrix[submatrix != 0]
        pairwise_sim_scores.append(relevant_elements.mean().item())
    avg_sim_scores[i] = mean(pairwise_sim_scores)
print(avg_sim_scores)

  0%|          | 0/199 [00:00<?, ?it/s]

{2: 0.7110402848571539, 3: 0.6958193264901638, 4: 0.6858824823051691, 5: 0.6870076607912778, 6: 0.6917016632854939, 7: 0.6950838375836611, 8: 0.6952130842953921, 9: 0.6947922468185425, 10: 0.696389428153634, 11: 0.6960095129907131, 12: 0.6961859870702028, 13: 0.6949956368654966, 14: 0.6950210381299258, 15: 0.6955876048654318, 16: 0.6949538048356771, 17: 0.6951644252985716, 18: 0.695039939135313, 19: 0.6946660611778498, 20: 0.694229182600975, 21: 0.6937927983701229, 22: 0.6933498196303844, 23: 0.6930181454867125, 24: 0.6924285747110843, 25: 0.6928547654300928, 26: 0.6932776045054198, 27: 0.6925003889948129, 28: 0.6924038849771023, 29: 0.6924892242997884, 30: 0.692736179754138, 31: 0.6925186105072498, 32: 0.691896291077137, 33: 0.6916094087064266, 34: 0.6920168690383435, 35: 0.6914364505559206, 36: 0.6914568439126014, 37: 0.6907116081565619, 38: 0.6906836375594139, 39: 0.6907121870666743, 40: 0.6909403417259454, 41: 0.6907816153019667, 42: 0.6909550655633211, 43: 0.6912170980125666, 44: 

In [ ]:
import plotly.graph_objects as go

smaller_list_keys = list(range(2,201)) # [3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 25, 50, 75, 100, 150, 200]
y = [avg_sim_scores[key] for key in smaller_list_keys]

# ...existing code...
fig = go.Figure()
fig.add_trace(go.Scatter(x=smaller_list_keys, y=y, mode='lines+markers'))
fig.update_layout(
    title="Average Similarity Scores",
    xaxis_title="#Questions",
    yaxis_title="Similarity",
    plot_bgcolor='rgba(0,0,0,0)',
    font=dict(size=26),
   
)
fig.update_xaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey', range=[0, 200])
fig.update_yaxes(showline=True, linewidth=2, linecolor='darkgrey', gridcolor='darkgrey')
fig.show()

fig.write_image('question_similarity_200.pdf', 'pdf', width=1300, height=600)